# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Kaggle notebook setup
### Installations

In [1]:
!pip install mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires

### Data location

In [2]:
import os
print(os.listdir('/kaggle/input/'))
print(os.listdir('/kaggle/input/radimagenet-densenet121-notop'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed/processed'))

['radimagenet-densenet121-notop', 'brain-tumor-mri-preprocessed']
['RadImageNet-DenseNet121_notop.h5']
['processed']
['Validation', 'Training', 'Testing', '.gitkeep']


## General

In [3]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("Brain_Tumor_Training")

<Experiment: artifact_location='mlflow-artifacts:/830660600119881173', creation_time=1769099937797, experiment_id='830660600119881173', last_update_time=1769099937797, lifecycle_stage='active', name='Brain_Tumor_Training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import math

2026-01-27 16:10:50.307919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769530250.496120      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769530250.554722      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769530251.060786      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769530251.060835      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769530251.060838      55 computation_placer.cc:177] computation placer alr

In [5]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Num GPUs Available: 1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [6]:
# path management
PROJECT_ROOT = '/kaggle/input'
PREP_DIR = PROJECT_ROOT + '/brain-tumor-mri-preprocessed/processed'
ARTEFACTS_DIR = PROJECT_ROOT + '/radimagenet-densenet121-notop'

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

PROJECT_NAME = "BrainTumorMRI"
MODEL_TYPE = "DenseNet121"
TWO_HEAD = True
MODEL_NAME = f"{PROJECT_NAME}_{MODEL_TYPE}_{TWO_HEAD*"2Head"}"
FREEZE_BACKBONE = True
MASK_TUMOR_TYPE_LOSS = True
BATCH_SIZE = 32
DATA_AUGMENTATION = True

## Modeling

### Backbone

In [7]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = not FREEZE_BACKBONE

print("✅ RadImageNet DenseNet121 loaded successfully")

I0000 00:00:1769530263.547229      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


✅ RadImageNet DenseNet121 loaded successfully


In [8]:
#backbone.summary()

In [9]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [211]:
model_data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomZoom((-0.03,0.03),(-0.03,0.03), seed=SEED),
    layers.RandomTranslation((-0.01,0.01),(-0.01,0.01), seed=SEED),
], name='data_augmentation_part')

In [212]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [213]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [214]:
def shared_head_part(inputs, backbone, data_augmentation):
    # Data augmentation (training only)
    x = data_augmentation(inputs)
    # Backbone - force into inference
    x = backbone(x, training=False)

    # Shared head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    return x

In [215]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = shared_head_part(inputs, backbone, model_data_augmentation)

#Heads
output_presence = model_head1(x)
output_type = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output_presence,
        "tumor_type": output_type
    },
    name='densenet_two_head'
)

In [216]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection, but taking account that tumors are 75% of data
)

In [217]:
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [218]:
model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },

    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": ["accuracy"],
    }
)

In [219]:
#model.summary()

## Streaming Training

In [220]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [221]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=BATCH_SIZE,\n    repeat=False\n).prefetch(tf.data.AUTOTUNE)\n'

In [222]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [223]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [224]:
#for x, y in train_ds.take(1):
#    print("Image:")
#    print(x.dtype, x.shape)
#    print("\nLabels:")
#    for k, v in y.items():
#        print(k, v.dtype, v.shape)

In [225]:
def count_tfrecord_batches(directory_path, batch_size):
    """
    Count number of batches for a TFRecord dataset.

    Assumes:
    - 1 sample per TFRecord
    - batch_size is variable

    Returns:
    - number of batches = ceil(num_samples / batch_size)
    """
    directory_path = Path(directory_path)
    n_samples = len(list(directory_path.glob("*.tfrecord")))
    return math.ceil(n_samples / batch_size)


In [226]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [227]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_tumor_presence_recall",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_tumor_presence_recall",
    mode="max",
    min_delta=0.0001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [228]:
#print("RUN_NAME:", RUN_NAME)
#print("TRAIN steps:", count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE))
#print("VAL steps:", count_tfrecord_batches(VAL_DIR, BATCH_SIZE))
#print("train_ds element_spec:", train_ds.element_spec)

In [229]:
RUN_NAME = (
    f"{MODEL_TYPE}"
    f"freeze={FREEZE_BACKBONE}_"
    f"mask={MASK_TUMOR_TYPE_LOSS}_"
    f"{datetime.now().strftime('%Y%m%d-%H%M')}"
)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.tensorflow.autolog(registered_model_name=MODEL_NAME)

    mlflow.log_params({
        "project": PROJECT_NAME,
        "model_type": MODEL_TYPE,
        "pretrained_weights": "RadImageNet",
        "backbone_frozen": FREEZE_BACKBONE,
        "head_1": "tumor_presence_binary",
        "head_2": "tumor_type_softmax",
        "mask_tumor_type_loss": MASK_TUMOR_TYPE_LOSS,
        "label_0_means": "no_tumor",
        "data_augmentation": DATA_AUGMENTATION,
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        #steps_per_epoch=count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE),
        #validation_steps=count_tfrecord_batches(VAL_DIR, BATCH_SIZE),
        callbacks=[reduce_lr, early_stopping],
        verbose=1,
    )

    client = MlflowClient()
    latest_version = client.get_latest_versions(MODEL_NAME)[-1].version

    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging"
    )


2026/01/27 18:50:20 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2026/01/27 18:50:22 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/50
    143/Unknown 37s 142ms/step - loss: 0.9327 - tumor_presence_accuracy: 0.8040 - tumor_presence_auc: 0.8375 - tumor_presence_loss: 0.1399 - tumor_presence_precision: 0.8727 - tumor_presence_recall: 0.8543 - tumor_type_accuracy: 0.5079 - tumor_type_loss: 0.7928

143/143 ━━━━━━━━━━━━━━━━━━━━ 62s 315ms/step - loss: 0.9315 - tumor_presence_accuracy: 0.8044 - tumor_presence_auc: 0.8379 - tumor_presence_loss: 0.1397 - tumor_presence_precision: 0.8728 - tumor_presence_recall: 0.8546 - tumor_type_accuracy: 0.5081 - tumor_type_loss: 0.7917 - val_loss: 1.4549 - val_tumor_presence_accuracy: 0.3841 - val_tumor_presence_auc: 0.9292 - val_tumor_presence_loss: 0.3882 - val_tumor_presence_precision: 0.9839 - val_tumor_presence_recall: 0.1481 - val_tumor_type_accuracy: 0.4217 - val_tumor_type_loss: 1.0262 - learning_rate: 0.0010
Epoch 2/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 175ms/step - loss: 0.5783 - tumor_presence_accuracy: 0.8893 - tumor_presence_auc: 0.9450 - tumor_presence_loss: 0.0746 - tumor_presence_precision: 0.9166 - tumor_presence_recall: 0.9314 - tumor_type_accuracy: 0.5705 - tumor_type_loss: 0.5037 - val_loss: 2.6564 - val_tumor_presence_accuracy: 0.5442 - val_tumor_presence_auc: 0.9307 - val_tumor_presence_loss: 0.2595 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 34s 237ms/step - loss: 0.5070 - tumor_presence_accuracy: 0.9189 - tumor_presence_auc: 0.9588 - tumor_presence_loss: 0.0635 - tumor_presence_precision: 0.9385 - tumor_presence_recall: 0.9500 - tumor_type_accuracy: 0.5929 - tumor_type_loss: 0.4434 - val_loss: 1.4117 - val_tumor_presence_accuracy: 0.4777 - val_tumor_presence_auc: 0.9392 - val_tumor_presence_loss: 0.3676 - val_tumor_presence_precision: 0.9749 - val_tumor_presence_recall: 0.2828 - val_tumor_type_accuracy: 0.4103 - val_tumor_type_loss: 1.0059 - learning_rate: 0.0010
Epoch 4/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 0.4700 - tumor_presence_accuracy: 0.9316 - tumor_presence_auc: 0.9710 - tumor_presence_loss: 0.0514 - tumor_presence_precision: 0.9462 - tumor_presence_recall: 0.9597 - tumor_type_accuracy: 0.5966 - tumor_type_loss: 0.4186

143/143 ━━━━━━━━━━━━━━━━━━━━ 42s 295ms/step - loss: 0.4701 - tumor_presence_accuracy: 0.9315 - tumor_presence_auc: 0.9710 - tumor_presence_loss: 0.0514 - tumor_presence_precision: 0.9461 - tumor_presence_recall: 0.9597 - tumor_type_accuracy: 0.5965 - tumor_type_loss: 0.4186 - val_loss: 0.7373 - val_tumor_presence_accuracy: 0.6063 - val_tumor_presence_auc: 0.9279 - val_tumor_presence_loss: 0.2327 - val_tumor_presence_precision: 0.9746 - val_tumor_presence_recall: 0.4660 - val_tumor_type_accuracy: 0.5652 - val_tumor_type_loss: 0.4855 - learning_rate: 0.0010
Epoch 5/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 174ms/step - loss: 0.4720 - tumor_presence_accuracy: 0.9310 - tumor_presence_auc: 0.9681 - tumor_presence_loss: 0.0531 - tumor_presence_precision: 0.9484 - tumor_presence_recall: 0.9564 - tumor_type_accuracy: 0.5953 - tumor_type_loss: 0.4190 - val_loss: 2.9879 - val_tumor_presence_accuracy: 0.8250 - val_tumor_presence_auc: 0.9485 - val_tumor_presence_loss: 0.1797 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 43s 295ms/step - loss: 0.4175 - tumor_presence_accuracy: 0.9404 - tumor_presence_auc: 0.9814 - tumor_presence_loss: 0.0417 - tumor_presence_precision: 0.9563 - tumor_presence_recall: 0.9612 - tumor_type_accuracy: 0.6077 - tumor_type_loss: 0.3758 - val_loss: 0.5462 - val_tumor_presence_accuracy: 0.9440 - val_tumor_presence_auc: 0.9760 - val_tumor_presence_loss: 0.0458 - val_tumor_presence_precision: 0.9481 - val_tumor_presence_recall: 0.9757 - val_tumor_type_accuracy: 0.5407 - val_tumor_type_loss: 0.4869 - learning_rate: 0.0010
Epoch 9/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 175ms/step - loss: 0.3971 - tumor_presence_accuracy: 0.9467 - tumor_presence_auc: 0.9806 - tumor_presence_loss: 0.0420 - tumor_presence_precision: 0.9562 - tumor_presence_recall: 0.9711 - tumor_type_accuracy: 0.6186 - tumor_type_loss: 0.3551 - val_loss: 1.1040 - val_tumor_presence_accuracy: 0.9484 - val_tumor_presence_auc: 0.9749 - val_tumor_presence_loss: 0.0476 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 38s 266ms/step - loss: 0.3322 - tumor_presence_accuracy: 0.9616 - tumor_presence_auc: 0.9896 - tumor_presence_loss: 0.0301 - tumor_presence_precision: 0.9750 - tumor_presence_recall: 0.9723 - tumor_type_accuracy: 0.6349 - tumor_type_loss: 0.3020 - val_loss: 0.5393 - val_tumor_presence_accuracy: 0.9615 - val_tumor_presence_auc: 0.9876 - val_tumor_presence_loss: 0.0305 - val_tumor_presence_precision: 0.9610 - val_tumor_presence_recall: 0.9867 - val_tumor_type_accuracy: 0.5363 - val_tumor_type_loss: 0.4960 - learning_rate: 5.0000e-04
Epoch 14/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 174ms/step - loss: 0.3271 - tumor_presence_accuracy: 0.9600 - tumor_presence_auc: 0.9904 - tumor_presence_loss: 0.0300 - tumor_presence_precision: 0.9734 - tumor_presence_recall: 0.9713 - tumor_type_accuracy: 0.6343 - tumor_type_loss: 0.2971 - val_loss: 0.5428 - val_tumor_presence_accuracy: 0.8801 - val_tumor_presence_auc: 0.9855 - val_tumor_presence_loss: 0.0712 - val_tumor_presence_pr

143/143 ━━━━━━━━━━━━━━━━━━━━ 39s 268ms/step - loss: 0.3271 - tumor_presence_accuracy: 0.9709 - tumor_presence_auc: 0.9933 - tumor_presence_loss: 0.0242 - tumor_presence_precision: 0.9797 - tumor_presence_recall: 0.9802 - tumor_type_accuracy: 0.6307 - tumor_type_loss: 0.3028 - val_loss: 0.4205 - val_tumor_presence_accuracy: 0.9493 - val_tumor_presence_auc: 0.9854 - val_tumor_presence_loss: 0.0376 - val_tumor_presence_precision: 0.9694 - val_tumor_presence_recall: 0.9600 - val_tumor_type_accuracy: 0.5879 - val_tumor_type_loss: 0.3731 - learning_rate: 5.0000e-04
Epoch 15: early stopping
Restoring model weights from the end of the best epoch: 5.
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


2026/01/27 18:58:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/27 18:58:55 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpaxf5f6o7/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
Registered model 'BrainTumorMRI_DenseNet121_2Head' already exists. Creating a new version of this model...
2026/01/27 18:59:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BrainTumorMRI_DenseNet121_2Head, version 11
Created version '11' of model 'BrainTumorMRI_DenseNet121_2Head'.
/tmp/ipykernel_55/1934392631.py:35: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecat

🏃 View run DenseNet121freeze=True_mask=True_20260127-1850 at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173/runs/bc8aafce13de489a9dca342c6e767e22
🧪 View experiment at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173


In [230]:
Warning : do not forget :
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning
- maximize reccal (y_pred > 0.3 → tumeur)

SyntaxError: invalid character '→' (U+2192) (1940605460.py, line 4)